<div align="center">

# Bring fse to your existing tools

Use one local encoder with your search, topic, and table libraries.

</div>

## When do you need this?

You already use a search library, a topic model, or a table library.
You want it to use the **same local encoder** as the rest of your app.
Choose a section below after running the shared setup, rather than installing a separate model for every tool.

## The story in pictures

![One POTION encoder supplies vectors to search libraries, a topic model, and a table column.](media/06-integrations.svg)

## 1. Prepare one encoder and real articles

The examples share POTION-base-8M and 100 [AG News](https://huggingface.co/datasets/SetFit/ag_news) articles.
**Embeddings** are vectors, lists of numbers that represent texts.
Each library receives those vectors explicitly, using the pinned versions from the [index](README.md#run-a-recipe).

In [1]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

Recipe run: 2026-11-01. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


Load POTION-base-8M and a few real articles first.
Every library below will use this encoder.

In [2]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr195-tutorials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Vector dimensions: 256


Use one small function wherever a library asks for vectors.
An existing article serves as the query to check that the connections return the intended row.

In [3]:
texts = recipe.news(100)


def embed(items):
    return encoder.encode(list(items), workers=1)


query = texts[0]
print("Example article:", query)

Example article: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.


## 2. Retrieve passages with DSPy

A **retriever** returns texts related to a query.
DSPy can call our local encoder directly, without a language model or a paid API.
We query with an existing article to **check the connection** before trying new search phrases.

In [4]:
import dspy

retriever = dspy.retrievers.Embeddings(corpus=texts, embedder=embed, k=1)
result = retriever(query)
assert result.passages == [texts[0]]
print("DSPy found:", result.passages[0])

DSPy found: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.


## 3. Group articles with BERTopic

**Topic modeling** groups documents and finds words that describe each group.
We provide vectors, use PCA to reduce their dimensions, then ask KMeans for four groups.
These are demonstration settings, so **inspect the topic words** before treating them as useful categories.

In [5]:
from bertopic import BERTopic
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

topic_model = BERTopic(
    embedding_model=None,
    umap_model=PCA(n_components=5),
    hdbscan_model=KMeans(n_clusters=4, random_state=7, n_init=10),
    vectorizer_model=CountVectorizer(stop_words="english"),
    calculate_probabilities=False,
)
topics, _ = topic_model.fit_transform(texts, embeddings=embed(texts))
assert len(topics) == len(texts) and len(set(topics)) == 4
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]].to_string(index=False))

W1101 09:25:08.487000 14568 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


 Topic  Count                         Name
     0     29   0_new_search_wireless_mars
     1     28       1_ap_england_day_night
     2     27     2_afp_najaf_reuters_said
     3     16 3_reuters_market_dollar_euro


## 4. Search with txtai

An **external transform** tells txtai to call your encoder instead of loading its own.
The callable accepts a batch of texts and returns one vector per text.

In [6]:
from txtai import Embeddings

index = Embeddings({"method": "external", "transform": embed, "content": True})
index.index([(i, text, None) for i, text in enumerate(texts)])
hits = index.search(query, 1)
assert hits[0]["id"] == "0" or hits[0]["id"] == 0
print("txtai:", hits)
index.close()

txtai: [{'id': '0', 'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'score': 1.0000001192092896}]


## 5. Store vectors in Chroma

Pass vectors when adding documents and when searching.
**Disable the default embedding function** so Chroma does not download another model.

In [7]:
import chromadb
from chromadb.config import Settings

chroma = chromadb.EphemeralClient(Settings(anonymized_telemetry=False))
collection = chroma.create_collection("fse-recipe", embedding_function=None)
collection.add(
    ids=[str(i) for i in range(len(texts))], documents=texts, embeddings=embed(texts).tolist()
)
hits = collection.query(query_embeddings=embed([query]).tolist(), n_results=1)
assert hits["ids"] == [["0"]]
print("Chroma:", hits["ids"])

Chroma: [['0']]


## 6. Search a local LanceDB table

Keep each article next to its vector in a temporary local table.
The query uses the same encoder, so stored and query vectors have the same meaning.

In [8]:
import lancedb

lance = lancedb.connect(str(work / "lance"))
table = lance.create_table(
    "texts",
    [
        {"id": i, "text": text, "vector": row.tolist()}
        for i, (text, row) in enumerate(zip(texts, embed(texts)))
    ],
)
hits = table.search(embed([query])[0]).limit(1).to_list()
assert hits[0]["id"] == 0
print("LanceDB:", hits[0]["text"])

LanceDB: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.


## 7. Write batches to Qdrant

A **sink** accepts vectors as fse produces them.
This small application adapter allocates a batch buffer, then upserts completed rows into local Qdrant.
Stable row IDs make replay replace existing points.
Memory-only Qdrant cannot promise crash durability, so this adapter reports no `durable_rows`.

In [9]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
client.create_collection(
    "texts",
    vectors_config=models.VectorParams(size=encoder.output_dim, distance=models.Distance.COSINE),
)


class QdrantSink:
    def __init__(self, client, collection, dim):
        self.client, self.collection, self.dim = client, collection, dim
        self.next_row = 0

    def target(self, first_row, n_rows):
        return np.empty((n_rows, self.dim), dtype=np.float32)

    def commit(self, first_row, rows):
        if first_row != self.next_row:
            raise ValueError("commits must be contiguous")
        self.client.upsert(
            self.collection,
            points=[
                models.PointStruct(id=first_row + i, vector=row.tolist())
                for i, row in enumerate(rows)
            ],
            wait=True,
        )
        self.next_row += len(rows)


sink: fse.Sink = QdrantSink(client, "texts", encoder.output_dim)
for batch in encoder.encode_batches([texts[:50], texts[50:]], sink=sink, workers=1):
    assert batch.durable_rows is None
assert client.count("texts", exact=True).count == len(texts)
print("Qdrant sink points:", client.count("texts", exact=True).count)
client.close()

Qdrant sink points: 100


## 8. Add vectors to a Polars table

A Polars **Series** is one column.
fse reads the text column through Arrow, the public table interchange interface, and returns a vector matrix.
Add it as a fixed-size array column, rather than making one Python call per row.

In [10]:
import polars as pl

frame = pl.DataFrame({"text": texts})
rows = encoder.encode(frame["text"], workers=1)
np.testing.assert_array_equal(rows, embed(texts))
frame = frame.with_columns(
    pl.Series("vector", rows, dtype=pl.Array(pl.Float32, encoder.output_dim))
)
assert frame.height == len(texts)
print(frame.head(3))

shape: (3, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ text                            ┆ vector                          │
│ ---                             ┆ ---                             │
│ str                             ┆ array[f32, 256]                 │
╞═════════════════════════════════╪═════════════════════════════════╡
│ Fears for T N pension after ta… ┆ [-0.067037, 0.164303, … 0.0219… │
│ The Race is On: Second Private… ┆ [0.054038, 0.121423, … 0.00471… │
│ Ky. Company Wins Grant to Stud… ┆ [0.235699, 0.157373, … -0.0556… │
└─────────────────────────────────┴─────────────────────────────────┘


## 9. Search with LangChain

**LangChain connects models and search tools.**
`langchain-fse` lets its vector stores call the encoder from our shared setup.
This section and the next two await fse 2.0.0 and publication of all three adapter packages.
The commands below describe installation after those releases, not packages currently available on PyPI.

```sh
pip install langchain-fse
```


An **in-memory vector store** keeps documents and their vectors in this Python process.
Add the articles, then search with the same article query used above.
No language model or API key is needed.


In [ ]:
from langchain_core.documents import Document as LangChainDocument
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_fse import FseEmbeddings

langchain_embeddings = FseEmbeddings(encoder=encoder, workers=1)
langchain_store = InMemoryVectorStore(langchain_embeddings)
langchain_store.add_documents([LangChainDocument(page_content=text) for text in texts])
langchain_hits = langchain_store.similarity_search(query, k=1)
assert langchain_hits[0].page_content == texts[0]
assert len(langchain_embeddings.embed_query(query)) == encoder.output_dim
print("LangChain found:", langchain_hits[0].page_content)


## 10. Retrieve articles with LlamaIndex

**LlamaIndex builds indexes of documents** and retrieves related passages.
`llama-index-embeddings-fse` supplies its embedding interface.
After publication, install it in the notebook's Python environment:

```sh
pip install llama-index-embeddings-fse
```


Pass the encoder explicitly so LlamaIndex uses our local POTION model.
An **index** stores the article vectors for search.
Use a retriever directly, rather than a query engine that would also need a language model.


In [ ]:
from llama_index.core import Document as LlamaDocument, VectorStoreIndex
from llama_index.embeddings.fse import FseEmbedding

llama_embeddings = FseEmbedding(encoder=encoder, workers=1, embed_batch_size=32)
llama_index = VectorStoreIndex.from_documents(
    [LlamaDocument(text=text) for text in texts], embed_model=llama_embeddings
)
llama_hits = llama_index.as_retriever(similarity_top_k=1).retrieve(query)
assert llama_hits[0].node.text == texts[0]
assert len(llama_embeddings.get_query_embedding(query)) == encoder.output_dim
print("LlamaIndex found:", llama_hits[0].node.text)


## 11. Connect a Haystack search pipeline

A Haystack **pipeline** connects components, each handling one step.
`fse-haystack` provides separate components for documents and query text.
After publication, install it in the notebook's Python environment:

```sh
pip install fse-haystack
```


First, encode the articles and write them to a local document store.
Call `warm_up()` before using an embedder by itself.
The supplied encoder is already loaded, so this needs no second model download.


In [ ]:
import os

os.environ["HAYSTACK_TELEMETRY_ENABLED"] = "false"

from haystack import Document as HaystackDocument, Pipeline
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack_integrations.components.embedders.fse import FseDocumentEmbedder, FseTextEmbedder

haystack_embedder = FseDocumentEmbedder(encoder=encoder, workers=1, batch_size=32)
haystack_embedder.warm_up()
haystack_documents = haystack_embedder.run(
    documents=[HaystackDocument(content=text) for text in texts]
)["documents"]
haystack_store = InMemoryDocumentStore(embedding_similarity_function="cosine")
haystack_store.write_documents(haystack_documents)
assert len(haystack_documents) == len(texts)
print("Haystack stored articles:", len(haystack_documents))


Next, connect query encoding to retrieval.
The pipeline passes the query vector to the retriever and returns the matching article.
These examples keep their stores in memory, so restarting Python removes the indexes.


In [ ]:
haystack_pipeline = Pipeline()
haystack_pipeline.add_component("embed", FseTextEmbedder(encoder=encoder, workers=1))
haystack_pipeline.add_component(
    "retrieve", InMemoryEmbeddingRetriever(document_store=haystack_store, top_k=1)
)
haystack_pipeline.connect("embed.embedding", "retrieve.query_embedding")
haystack_hits = haystack_pipeline.run({"embed": {"text": query}})["retrieve"]["documents"]
assert haystack_hits[0].content == texts[0]
assert len(haystack_documents[0].embedding) == encoder.output_dim
print("Haystack found:", haystack_hits[0].content)


## Result

The search examples return the article used as the query, and BERTopic reports four groups.
LangChain, LlamaIndex and Haystack each retrieve the same article through their fse package.
The assertions check that each library receives and stores **the intended rows**.
This checks these connections for one example, rather than measuring search quality.


## What next?

- [Handle queue replay before copying to Qdrant](04-kafka-and-qdrant.ipynb).
- [fse API and design](../docs/architecture/runtime.md)
- [DSPy retriever](https://github.com/stanfordnlp/dspy/blob/main/dspy/retrievers/embeddings.py), [BERTopic vectors](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html), [txtai configuration](https://neuml.github.io/txtai/embeddings/configuration/vectors/), [Chroma](https://docs.trychroma.com/docs/embeddings/embedding-functions), [LanceDB](https://docs.lancedb.com/search/vector-search), [Qdrant](https://github.com/qdrant/qdrant-client), [Polars](https://docs.pola.rs/user-guide/expressions/user-defined-python-functions/)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [11]:
recipe.close()